# Phase B: Phase 2 Diagnostic Sweep

**7 experiments** to diagnose root cause of Phase 2 instability.


## 1. Environment Setup


In [ ]:
# Install dependencies
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy glada package from Drive to local SSD (faster imports)
import shutil, os, sys
DRIVE_PKG = "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code"
LOCAL_PKG = '/content/glada_code'
if os.path.exists(LOCAL_PKG):
    shutil.rmtree(LOCAL_PKG)
shutil.copytree(DRIVE_PKG, LOCAL_PKG)
print(f'Package copied to {LOCAL_PKG}')

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
else:
    print('WARNING: No GPU detected')


### 1.1 Dataset Extraction

In [ ]:
import os
import shutil
import tarfile
import zipfile

DRIVE_DATASETS = '/content/drive/MyDrive/Jakob/Datasets'
LOCAL_DATA = '/content/data'


def extract_one_dataset(drive_archive, extract_to, verify_path, name):
    """Copy archive from Drive -> runtime SSD, extract, delete archive."""
    if os.path.isdir(verify_path):
        print(f'[{name}] Already extracted.')
        return
    os.makedirs(extract_to, exist_ok=True)
    local_archive = os.path.join(LOCAL_DATA, os.path.basename(drive_archive))
    print(f'[{name}] Copying archive to SSD...')
    shutil.copy2(drive_archive, local_archive)
    print(f'[{name}] Extracting...')
    if local_archive.endswith('.zip'):
        with zipfile.ZipFile(local_archive, 'r') as zf:
            zf.extractall(extract_to)
    elif local_archive.endswith(('.tar.gz', '.tgz')):
        with tarfile.open(local_archive, 'r:gz') as tf:
            tf.extractall(extract_to)
    os.remove(local_archive)
    print(f'[{name}] Done. Archive deleted from SSD.')


# ASVspoof 2019 LA
extract_one_dataset(
    f'{DRIVE_DATASETS}/LA.zip', f'{LOCAL_DATA}/asvspoof',
    f'{LOCAL_DATA}/asvspoof/LA', 'ASVspoof')

# In-the-Wild
extract_one_dataset(
    f'{DRIVE_DATASETS}/release_in_the_wild.zip', f'{LOCAL_DATA}/itw',
    f'{LOCAL_DATA}/itw/release_in_the_wild', 'ITW')

# MUSAN
extract_one_dataset(
    f'{DRIVE_DATASETS}/musan.tar.gz', LOCAL_DATA,
    f'{LOCAL_DATA}/musan', 'MUSAN')

# RIR
extract_one_dataset(
    f'{DRIVE_DATASETS}/rirs_noises.zip', LOCAL_DATA,
    f'{LOCAL_DATA}/RIRS_NOISES/simulated_rirs', 'RIR')

print('\nAll datasets extracted and verified.')

In [ ]:
from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed

from glada.datasets import (
    ASVspoofDatasetWithSpeaker,
    ASVspoofEvalDataset,
    InTheWildDataset,
    create_dataloaders,
)
from glada.augmentation import RawBoost, CombinedAugmenter
from glada.trainers import GLADATrainer

import torch
import json
import gc

print(f'GLADA package loaded.')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')

## 2. Sweep Configuration

In [ ]:
# ============================================================
# SWEEP CONFIGURATION
# Each trial runs Phase 1 + a few Phase 2 epochs to test backbone fine-tuning.
# Ranked by stability-aware score (last_p2_eer + penalty for degradation trend).
# Phase 2 adversarial ramp prevents gradient shock at backbone unfreeze.
# ============================================================

TRIAL_PHASE2_EPOCHS = 5   # Phase 2 epochs per trial (enough to see degradation)
SEED = 42
BACKBONE = 'wavlm-large'

# --- Dataset paths ---
ASV_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_train/flac'
ASV_EVAL_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
ASV_EVAL_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
ITW_META = '/content/data/itw/release_in_the_wild/meta.csv'
ITW_AUDIO = '/content/data/itw/release_in_the_wild'
MUSAN_PATH = '/content/data/musan'
RIR_PATH = '/content/data/RIRS_NOISES/simulated_rirs'

# --- Phase 2 adversarial ramp (gradient shock prevention) ---
# These are the new defaults from config.py; made explicit here for visibility.
PHASE2_ADV_WARMUP_EPOCHS = 3   # Ramp GRL alphas from 0 over 3 Phase 2 epochs
PHASE2_ADV_RAMP_START = 0.0    # Full reset at Phase 2 onset
PHASE2_CKA_RAMP = True         # Also ramp CKA weight during Phase 2 warmup

# --- Sweep grid ---
SWEEP_CONFIGS = [
    # (backbone_lr, freeze_bottom_n, phase1_epochs, label)
    (5e-6,  0,  5, 'lr5e6_freeze0_p1_5'),
    (2e-6,  0,  5, 'lr2e6_freeze0_p1_5'),
    (1e-6,  0,  5, 'lr1e6_freeze0_p1_5'),
    (5e-6, 12,  5, 'lr5e6_freeze12_p1_5'),
    (2e-6, 12,  5, 'lr2e6_freeze12_p1_5'),
    (1e-6, 12,  5, 'lr1e6_freeze12_p1_5'),
    (5e-6, 18,  5, 'lr5e6_freeze18_p1_5'),
    (2e-6, 18,  5, 'lr2e6_freeze18_p1_5'),
    (5e-6, 12, 10, 'lr5e6_freeze12_p1_10'),
    (2e-6, 12, 10, 'lr2e6_freeze12_p1_10'),
]

print(f'Sweep: {len(SWEEP_CONFIGS)} configs x '
      f'(Phase1 + {TRIAL_PHASE2_EPOCHS} Phase2 epochs each)')
print(f'Phase 2 adv ramp: {PHASE2_ADV_RAMP_START} -> 1.0 over '
      f'{PHASE2_ADV_WARMUP_EPOCHS} P2 epochs (CKA ramp={PHASE2_CKA_RAMP})')
for lr, freeze, p1, label in SWEEP_CONFIGS:
    total = p1 + TRIAL_PHASE2_EPOCHS
    print(f'  {label}: backbone_lr={lr}, freeze_bottom={freeze}, '
          f'phase1={p1}, total={total} ep')

## 3. Run Sweep

In [ ]:
sweep_results = []

for i, (backbone_lr, freeze_n, p1_epochs, label) in enumerate(SWEEP_CONFIGS):
    print(f'\n{"="*60}')
    print(f'TRIAL {i+1}/{len(SWEEP_CONFIGS)}: {label}')
    print(f'{"="*60}')

    # --- Build config for this trial ---
    config = GLADAConfig(
        experiment_name=f'sweep_{label}',
        backbone=BACKBONE,
        seed=SEED,

        # Trial-specific
        backbone_lr=backbone_lr,
        freeze_bottom_n_layers=freeze_n,
        phase1_epochs=p1_epochs,
        phase2_epochs=TRIAL_PHASE2_EPOCHS,

        # Phase 2 adversarial ramp (gradient shock prevention)
        phase2_adv_warmup_epochs=PHASE2_ADV_WARMUP_EPOCHS,
        phase2_adv_ramp_start=PHASE2_ADV_RAMP_START,
        phase2_cka_ramp=PHASE2_CKA_RAMP,

        # Fixed (same as main experiment)
        batch_size=24,
        effective_batch_size=384,
        head_lr=1e-3,
        vib_beta=1e-2,
        domain_loss_weight=0.1,
        speaker_loss_weight=0.1,
        cka_loss_weight=0.01,
        rawboost_prob=0.5,
        musan_path=MUSAN_PATH,
        musan_prob=0.5,
        rir_path=RIR_PATH,
        rir_prob=0.5,

        # Ablation switches (full model)
        use_glada=True,
        use_vib=True,
        use_speaker_adversary=True,
        use_cka_loss=True,
        use_hierarchical=True,
        use_rawboost=True,

        # Lighter checkpointing for trials
        save_every_n_epochs=0,
        save_top_k=1,
        patience=999,  # No early stopping during sweep trials
    )

    set_seed(SEED)

    # --- Build datasets and loaders (fresh each trial) ---
    augmenter = CombinedAugmenter(config)

    source_ds = ASVspoofDatasetWithSpeaker(
        protocol_path=ASV_PROTO,
        audio_dir=ASV_AUDIO,
        config=config,
        augmenter=augmenter,
    )
    target_ds = InTheWildDataset(
        meta_csv_path=ITW_META,
        audio_dir=ITW_AUDIO,
        config=config,
        use_labels=False,
    )
    eval_ds = InTheWildDataset(
        meta_csv_path=ITW_META,
        audio_dir=ITW_AUDIO,
        config=config,
        use_labels=True,
    )
    extra_eval = {}
    if os.path.exists(ASV_EVAL_PROTO) and os.path.exists(ASV_EVAL_AUDIO):
        extra_eval['2019_LA'] = ASVspoofEvalDataset(
            protocol_path=ASV_EVAL_PROTO,
            audio_dir=ASV_EVAL_AUDIO,
            config=config,
        )

    loaders = create_dataloaders(
        config=config,
        source_ds=source_ds,
        target_ds=target_ds,
        eval_ds=eval_ds,
        extra_eval=extra_eval,
    )
    source_loader = loaders['source']
    target_loader = loaders['target']
    eval_loader = loaders['eval']
    extra_eval_loaders = {k: v for k, v in loaders.items()
                          if k not in ('source', 'target', 'eval')}

    # --- Train ---
    trainer = GLADATrainer(
        config=config,
        source_loader=source_loader,
        target_loader=target_loader,
        eval_loader=eval_loader,
        extra_eval_loaders=extra_eval_loaders,
        num_speakers=source_ds.num_speakers,
    )
    results = trainer.train()

    # --- Extract Phase 2 metrics ---
    phase2_eers = []
    for entry in results['history']:
        if entry['phase'] == 2:
            phase2_eers.append(entry['eval']['ITW']['eer'])

    best_p2_eer = min(phase2_eers) if phase2_eers else 999.0
    last_p2_eer = phase2_eers[-1] if phase2_eers else 999.0

    # Phase 2 trend: positive = degrading, negative = improving
    if len(phase2_eers) >= 2:
        trend = phase2_eers[-1] - phase2_eers[0]
    else:
        trend = 0.0

    # Stability-aware score: penalises configs that degrade during Phase 2
    stability_score = last_p2_eer + 0.5 * max(0.0, trend)

    trial_result = {
        'label': label,
        'backbone_lr': backbone_lr,
        'freeze_bottom_n': freeze_n,
        'phase1_epochs': p1_epochs,
        'best_itw_eer': results['best_eer'],
        'best_epoch': results['best_epoch'],
        'best_p2_eer': best_p2_eer,
        'last_p2_eer': last_p2_eer,
        'p2_trend': trend,
        'stability_score': stability_score,
        'phase2_eers': phase2_eers,
        'p2_adv_ramp': [e.get('p2_adv_ramp_factor', 1.0)
                        for e in results['history'] if e['phase'] == 2],
    }
    sweep_results.append(trial_result)

    trend_arrow = '\u2191' if trend > 0 else '\u2193' if trend < 0 else '\u2192'
    print(f'\n  >> {label}: Best={results["best_eer"]:.2f}% (ep{results["best_epoch"]}), '
          f'P2 last={last_p2_eer:.2f}%, trend={trend:+.2f}% {trend_arrow}, '
          f'score={stability_score:.2f}')

    # --- Cleanup GPU memory ---
    del trainer, results, source_ds, target_ds, eval_ds, loaders
    del source_loader, target_loader, eval_loader, extra_eval_loaders
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('SWEEP COMPLETE')
print(f'{"="*60}')

## 4. Results Summary

In [ ]:
# --- Print results table ---
print(f'{"Config":<30} {"Score":>10} {"Last P2":>10} {"P2 Trend":>10} {"Best P2":>10} {"Ramp":>8}')
print('-' * 85)

# Sort by stability score (not best_p2_eer — that rewards lucky early epochs)
ranked = sorted(sweep_results, key=lambda x: x['stability_score'])

for r in ranked:
    trend_arrow = '\u2191' if r['p2_trend'] > 0 else '\u2193' if r['p2_trend'] < 0 else '\u2192'
    ramp_str = ','.join(f'{v:.2f}' for v in r.get('p2_adv_ramp', []))
    print(f'{r["label"]:<30} {r["stability_score"]:>9.2f}  {r["last_p2_eer"]:>9.2f}% '
          f'{r["p2_trend"]:>+9.2f}{trend_arrow} {r["best_p2_eer"]:>9.2f}% {ramp_str:>8}')

winner = ranked[0]
print(f'\n*** WINNER (by stability score): {winner["label"]} ***')
print(f'    backbone_lr={winner["backbone_lr"]}, '
      f'freeze_bottom={winner["freeze_bottom_n"]}, '
      f'phase1_epochs={winner["phase1_epochs"]}')
print(f'    Stability score: {winner["stability_score"]:.2f}')
print(f'    Last Phase 2 EER: {winner["last_p2_eer"]:.2f}%, '
      f'Trend: {winner["p2_trend"]:+.2f}%')
print(f'    Phase 2 EERs: {[f"{e:.2f}" for e in winner["phase2_eers"]]}')

# --- Save results to Drive ---
sweep_path = '/content/drive/MyDrive/Jakob/Experiments/sweep_results.json'
with open(sweep_path, 'w') as f:
    json.dump(ranked, f, indent=2)
print(f'\nResults saved to {sweep_path}')

## 5. Full Training with Winner

Runs the winning config for a full 50 epochs. Alternatively, copy the values
into `glada_experiment.ipynb` to run separately.

In [ ]:
# --- Full training with the winning config ---
w = winner  # from the sweep results above

print(f'Running full training with: {w["label"]}')
print(f'  backbone_lr={w["backbone_lr"]}, freeze_bottom={w["freeze_bottom_n"]}, '
      f'phase1_epochs={w["phase1_epochs"]}')

config = GLADAConfig(
    experiment_name=f'glada_{BACKBONE}_seed{SEED}',
    backbone=BACKBONE,
    seed=SEED,

    # --- Winner config ---
    backbone_lr=w['backbone_lr'],
    freeze_bottom_n_layers=w['freeze_bottom_n'],
    phase1_epochs=w['phase1_epochs'],
    phase2_epochs=45,  # Full Phase 2

    # Phase 2 adversarial ramp (same as sweep)
    phase2_adv_warmup_epochs=PHASE2_ADV_WARMUP_EPOCHS,
    phase2_adv_ramp_start=PHASE2_ADV_RAMP_START,
    phase2_cka_ramp=PHASE2_CKA_RAMP,

    # --- Fixed params ---
    batch_size=24,
    effective_batch_size=384,
    head_lr=1e-3,
    vib_beta=1e-2,
    domain_loss_weight=0.1,
    speaker_loss_weight=0.1,
    cka_loss_weight=0.01,
    rawboost_prob=0.5,
    musan_path=MUSAN_PATH,
    musan_prob=0.5,
    rir_path=RIR_PATH,
    rir_prob=0.5,

    use_glada=True,
    use_vib=True,
    use_speaker_adversary=True,
    use_cka_loss=True,
    use_hierarchical=True,
    use_rawboost=True,

    save_every_n_epochs=1,
    save_top_k=10,
)

set_seed(SEED)

augmenter = CombinedAugmenter(config)

source_ds = ASVspoofDatasetWithSpeaker(
    protocol_path=ASV_PROTO,
    audio_dir=ASV_AUDIO,
    config=config,
    augmenter=augmenter,
)
target_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=config,
    use_labels=False,
)
eval_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=config,
    use_labels=True,
)
extra_eval = {}
if os.path.exists(ASV_EVAL_PROTO) and os.path.exists(ASV_EVAL_AUDIO):
    extra_eval['2019_LA'] = ASVspoofEvalDataset(
        protocol_path=ASV_EVAL_PROTO,
        audio_dir=ASV_EVAL_AUDIO,
        config=config,
    )

loaders = create_dataloaders(
    config=config,
    source_ds=source_ds,
    target_ds=target_ds,
    eval_ds=eval_ds,
    extra_eval=extra_eval,
)

trainer = GLADATrainer(
    config=config,
    source_loader=loaders['source'],
    target_loader=loaders['target'],
    eval_loader=loaders['eval'],
    extra_eval_loaders={k: v for k, v in loaders.items()
                        if k not in ('source', 'target', 'eval')},
    num_speakers=source_ds.num_speakers,
)

results = trainer.train()
print(f'\nFinal Best ITW EER: {results["best_eer"]:.2f}% (epoch {results["best_epoch"]})')

---

## 6. Phase 2 Diagnostic Sweep (Dr. Abri's Request)

7 controlled experiments to isolate the root cause of Phase 2 instability.
Answers: is it the adversarial signals, the ramp, or something else?

| # | Config | What it tests |
|---|--------|---------------|
| 1 | `P2_diag_control` | Bug-fixed baseline with ramp |
| 2 | `P2_diag_no_ramp` | Is ramp still needed post-fix? |
| 3 | `P2_diag_glada_only` | GLADA discriminators only (no speaker) |
| 4 | `P2_diag_speaker_only` | Speaker adversary only (no GLADA) |
| 5 | `P2_diag_no_adversarial` | No adversarial at all — baseline |
| 6 | `P2_diag_ramp5` | Longer ramp (5 epochs) |
| 7 | `P2_diag_ramp3_start05` | Start at half-strength |


In [ ]:
from glada.trainers.sweep import generate_phase2_diagnostic_configs

diag_configs = generate_phase2_diagnostic_configs(
    batch_size=64,
    amp_dtype='bfloat16',
)

print(f'{len(diag_configs)} diagnostic configs:')
for cfg in diag_configs:
    print(f'  {cfg.experiment_name}: glada={cfg.use_glada}, speaker={cfg.use_speaker_adversary}, '
          f'ramp={cfg.phase2_adv_warmup_epochs}, ramp_start={cfg.phase2_adv_ramp_start}')


In [ ]:
diag_results = []

for i, config in enumerate(diag_configs):
    print(f'\n{"="*60}')
    print(f'DIAGNOSTIC {i+1}/{len(diag_configs)}: {config.experiment_name}')
    print(f'{"="*60}')

    # Override paths
    config.musan_path = MUSAN_PATH
    config.rir_path = RIR_PATH

    set_seed(config.seed)
    augmenter = CombinedAugmenter(config)

    source_ds = ASVspoofDatasetWithSpeaker(
        protocol_path=ASV_PROTO,
        audio_dir=ASV_AUDIO,
        config=config,
        augmenter=augmenter,
    )
    target_ds = InTheWildDataset(
        meta_csv_path=ITW_META,
        audio_dir=ITW_AUDIO,
        config=config,
        use_labels=False,
    )
    eval_ds = InTheWildDataset(
        meta_csv_path=ITW_META,
        audio_dir=ITW_AUDIO,
        config=config,
        use_labels=True,
    )
    extra_eval = {}
    if os.path.exists(ASV_EVAL_PROTO) and os.path.exists(ASV_EVAL_AUDIO):
        extra_eval['2019_LA'] = ASVspoofEvalDataset(
            protocol_path=ASV_EVAL_PROTO,
            audio_dir=ASV_EVAL_AUDIO,
            config=config,
        )

    loaders = create_dataloaders(
        config=config,
        source_ds=source_ds,
        target_ds=target_ds,
        eval_ds=eval_ds,
        extra_eval=extra_eval,
    )

    trainer = GLADATrainer(
        config=config,
        source_loader=loaders['source'],
        target_loader=loaders['target'],
        eval_loader=loaders['eval'],
        extra_eval_loaders={k: v for k, v in loaders.items()
                            if k not in ('source', 'target', 'eval')},
        num_speakers=source_ds.num_speakers,
    )
    results = trainer.train()

    # Extract per-phase EERs
    p1_eers = [e['eval']['ITW']['eer'] for e in results['history'] if e['phase'] == 1]
    p2_eers = [e['eval']['ITW']['eer'] for e in results['history'] if e['phase'] == 2]
    p2_trend = (p2_eers[-1] - p2_eers[0]) if len(p2_eers) >= 2 else 0.0

    diag_results.append({
        'name': config.experiment_name,
        'use_glada': config.use_glada,
        'use_speaker': config.use_speaker_adversary,
        'ramp_epochs': config.phase2_adv_warmup_epochs,
        'ramp_start': config.phase2_adv_ramp_start,
        'best_eer': results['best_eer'],
        'best_epoch': results['best_epoch'],
        'p1_final_eer': p1_eers[-1] if p1_eers else None,
        'p2_eers': p2_eers,
        'p2_trend': p2_trend,
        'stable': p2_trend <= 0,
    })

    stable = 'STABLE' if p2_trend <= 0 else 'UNSTABLE'
    print(f'  >> {config.experiment_name}: P1_final={p1_eers[-1]:.2f}%, '
          f'P2_last={p2_eers[-1]:.2f}%, trend={p2_trend:+.2f}% [{stable}]')

    del trainer, results, source_ds, target_ds, eval_ds, loaders
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('DIAGNOSTIC SWEEP COMPLETE')
print(f'{"="*60}')


## 7. Diagnostic Results


In [ ]:
# --- Diagnostic results table ---
print(f'{"Config":<30} {"GLADA":>6} {"SPK":>5} {"Ramp":>5} {"P1 End":>8} {"P2 Last":>8} {"Trend":>8} {"Status":>8}')
print('-' * 90)
for r in diag_results:
    trend_arrow = '\u2191' if r['p2_trend'] > 0 else '\u2193' if r['p2_trend'] < 0 else '\u2192'
    status = 'STABLE' if r['stable'] else 'UNSTABLE'
    p1_str = f'{r["p1_final_eer"]:.2f}%' if r['p1_final_eer'] else 'N/A'
    p2_last = r['p2_eers'][-1] if r['p2_eers'] else 0
    print(f'{r["name"]:<30} {str(r["use_glada"]):>6} {str(r["use_speaker"]):>5} '
          f'{r["ramp_epochs"]:>5} {p1_str:>8} {p2_last:>7.2f}% {r["p2_trend"]:>+7.2f}{trend_arrow} {status:>8}')

# --- Interpretation ---
print('\n--- Interpretation Guide ---')
print('Compare configs 3 vs 4 vs 5 to identify which adversarial signal causes instability:')
print('  - If config 5 (no adversarial) is stable but 3/4 are not -> adversarial-specific')
print('  - If config 3 (GLADA only) is unstable but 4 (speaker only) is stable -> GLADA causes it')
print('  - If config 4 (speaker only) is unstable but 3 (GLADA only) is stable -> speaker causes it')
print('Compare configs 1 vs 2 to check if ramp is still needed after speaker scaling fix:')
print('  - If both stable -> ramp no longer needed')
print('  - If 2 unstable but 1 stable -> ramp still needed')

# --- Save ---
diag_path = '/content/drive/MyDrive/Jakob/Experiments/phase2_diagnostic_results.json'
with open(diag_path, 'w') as f:
    json.dump(diag_results, f, indent=2)
print(f'\nResults saved to {diag_path}')
